## Learning notes — 06. COPY INTO And MERGE Commands

### Learning objectives
Follow **06. COPY INTO And MERGE Commands** from its source inputs through its operations to the archived results.
- Explain merge, autoloader, schema, write, profiling as used in this lesson.

### Source-specific learning focus
- Trace key-based changes from incoming records into a persistent target.
- Separate file discovery, parsing and schema tracking.
- Compare raw fields with typed records and parsing failures.
- Trace how source records change persistent destination state.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.

SQL relation identifiers referenced in source (inputs and/or destinations; not a claim that these objects exist locally): `FILEFORMAT`, `SET`, `demo.delta_lake`, `demo.delta_lake.raw_stock_prices`, `demo.delta_lake.stock_prices`.


## Learning notes — supplied input snapshot

Path: `data/stock_prices/stock_prices_day1.json`

SHA-256: `fa5587aa3a3844d9b6a344e13c691e750e72b16ac5a95c97cc8d29995306101d`

Complete supplied text (not generated output):

```json
{"stock_id":"AAPL","trading_date":"2025-02-10","price":227.65,"status":"ACTIVE"}
{"stock_id":"GOOGL","trading_date":"2025-02-10","price":2775.0,"status":"ACTIVE"}
{"stock_id":"MSFT","trading_date":"2025-02-10","price":325.0,"status":"ACTIVE"}
```

## Learning notes — supplied input snapshot

Path: `data/stock_prices/stock_prices_day2.json`

SHA-256: `756bbae2da3113c44d94e4b4862899cc69de371747e26b78a35d2f1fbbe07b5c`

Complete supplied text (not generated output):

```json
{"stock_id":"AMZN","trading_date":"2025-02-11","price":3475.0,"status":"ACTIVE"}
{"stock_id":"TSLA","trading_date":"2025-02-11","price":155.0,"status":"ACTIVE"}
```

## Learning notes — supplied input snapshot

Path: `data/stock_prices/stock_prices_day3.json`

SHA-256: `bedaa05d4909b2e605cb6811802d928f68e2e802092c0ebf9ff4ca945ecca11d`

Complete supplied text (not generated output):

```json
{"stock_id":"AAPL","trading_date":"2025-02-12","price":233.5,"status":"ACTIVE"}
{"stock_id":"GOOGL","trading_date":"2025-02-12","price":2805.0,"status":"ACTIVE"}
{"stock_id":"MSFT","trading_date":"2025-02-12","price":335.0,"status":"ACTIVE"}
{"stock_id":"AMZN","trading_date":"2025-02-12","price":3520.0,"status":"ACTIVE"}
{"stock_id":"META","trading_date":"2025-02-12","price":405.0,"status":"ACTIVE"}
{"stock_id":"TSLA","trading_date":"2025-02-12","price":null,"status":"DELISTED"}
```

# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

# COPY INTO and MERGE Commands

### COPY INTO Command

> - Incrementally loads data into Delta Lake tables from Cloud Storage  
> - Supports schema evolution  
> - Supports wide range of file formats (CSV, JSON, Parquet, Delta)  
> - Alternative to Auto Loader for batch ingestion  

Documentation - [COPY INTO](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-copy-into)

#### Create the table to copy the data into

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.raw_stock_prices;
```

In [ ]:
# Archived output only; no computation.

Archived table (empty):

#### Incrementally load new files into the table

## Learning notes — before source operation

**schema**: Compare declared or inferred field types with raw input. Parsing, schema evolution and rescued/corrupt records can produce nulls or additional columns rather than clean typed rows.

**Unexecuted teaching example (illustrative, not an archived or generated result):** Parsing a numeric field from an incompatible string can fail or yield null depending on parsing mode; inspect the declared schema and error policy.

**write**: This operation changes persistent state. Identify append, overwrite or incremental-copy semantics before rerunning; archived output does not establish the current destination contents.

Archived Databricks source — shown for reading, not Python execution.

```sql
DELETE FROM demo.delta_lake.raw_stock_prices;

COPY INTO demo.delta_lake.raw_stock_prices
FROM 'abfss://demo@deacourseextdl.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');
```

In [ ]:
# Archived output only; no computation.

Archived table:
num_affected_rows	num_inserted_rows	num_skipped_corrupt_files
6	6	0

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM demo.delta_lake.raw_stock_prices;
```

In [ ]:
# Archived output only; no computation.

Archived table:
price	status	stock_id	trading_date
233.5	ACTIVE	AAPL	2025-02-12
2805.0	ACTIVE	GOOGL	2025-02-12
335.0	ACTIVE	MSFT	2025-02-12
3520.0	ACTIVE	AMZN	2025-02-12
405.0	ACTIVE	META	2025-02-12
None	DELISTED	TSLA	2025-02-12

### MERGE Statement
> - Used for upserts (Insert/ Update/ Delete operations in a single statement)
> - Allows merging new data into a target table based on matching condition

Documentation - [MERGE INTO](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-merge-into)

#### Create the table to merge the data into

Archived Databricks source — shown for reading, not Python execution.

```sql
CREATE TABLE IF NOT EXISTS demo.delta_lake.stock_prices
(
  stock_id STRING,
  price DOUBLE,
  trading_date DATE
);
```

In [ ]:
# Archived output only; no computation.

Archived table (empty):

#### Merge the source data into target table
1. Insert new stocks received
2. Update price and trading_date if updates receieved
3. Delete stocks which are de-listed from the exchange (status = 'DELISTED')

## Learning notes — before source operation

**merge**: MERGE matches source rows to target keys and applies matched/unmatched actions. Verify source-key uniqueness and distinguish updates from inserts.

**write**: This operation changes persistent state. Identify append, overwrite or incremental-copy semantics before rerunning; archived output does not establish the current destination contents.

Archived Databricks source — shown for reading, not Python execution.

```sql
MERGE INTO demo.delta_lake.stock_prices AS target
USING demo.delta_lake.raw_stock_prices AS source
   ON target.stock_id = source.stock_id
WHEN MATCHED AND source.status = 'ACTIVE' THEN
    UPDATE SET target.price = source.price, target.trading_date = source.trading_date
WHEN MATCHED AND source.status = 'DELISTED' THEN    
    DELETE
WHEN NOT MATCHED AND source.status ='ACTIVE' THEN     
    INSERT (stock_id, price, trading_date) VALUES (source.stock_id, source.price, source.trading_date);
```

In [ ]:
# Archived output only; no computation.

Archived table:
num_affected_rows	num_updated_rows	num_deleted_rows	num_inserted_rows
6	4	1	1

## Learning notes — before source operation

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

Archived Databricks source — shown for reading, not Python execution.

```sql
SELECT * FROM demo.delta_lake.stock_prices;
```

In [ ]:
# Archived output only; no computation.

Archived table:
stock_id	price	trading_date
AAPL	233.5	2025-02-12
GOOGL	2805.0	2025-02-12
MSFT	335.0	2025-02-12
AMZN	3520.0	2025-02-12
META	405.0	2025-02-12